# Model search: *which* constraints are active?

Every other MVR notebook fixes a constraint set and asks about hidden paths. This one
fixes the data and asks about the constraint set. Given a pool of candidate constraints
$\mathcal{C}$ and one observed sequence $D$ drawn from $P(\cdot \mid A^*)$ for an
unknown active set $A^* \subseteq \mathcal{C}$, which subset best explains $D$?

The model space is $2^{|\mathcal{C}|}$, so we search rather than enumerate, using
**Shotgun Stochastic Search** (Hans, Dobra and West, 2007) driven by an exact Bayes
factor. For a base set $B$ and a candidate $C \notin B$,

$$\log \frac{P(D \mid B \cup \{C\})}{P(D \mid B)}
  \;=\; \log \frac{P(C \mid D,\, B)}{P(C \mid B)}$$

The gain from adding a constraint is the ratio of its **posterior** satisfaction
probability to its **prior** one — and `sat_prob_mvr` already computes both. A
constraint earns its place exactly when the data make it *more likely to hold* than the
model alone does.

In [ ]:
import itertools
import math

import matplotlib.pyplot as plt
import numpy as np
import torch

from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.mvr import HomMVR
from conin.hidden_markov_model.chmm_mvr import MVR_CHMM
from conin.hidden_markov_model.sampling.ffbs_mvr import ffbs_torch_mvr_chmm
from conin.hidden_markov_model.sss_mvr import (
    bayes_factor_mvr_chmm,
    log_evidence_mvr_chmm,
    sss_torch_mvr_chmm,
)


HIDDEN_STATES = ["A", "B", "C"]
OBSERVED_STATES = ["lo", "mid", "hi"]

hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs={
        "A": 0.2765440507007986,
        "B": 0.4033576072467887,
        "C": 0.32009834205241255,
    },
    transition_probs={
        ("A", "A"): 0.3391777054270445,
        ("A", "B"): 0.049711711669595204,
        ("A", "C"): 0.6111105829033604,
        ("B", "A"): 0.48102507253852517,
        ("B", "B"): 0.05601918704283972,
        ("B", "C"): 0.4629557404186351,
        ("C", "A"): 0.43616112524444134,
        ("C", "B"): 0.1773076392327265,
        ("C", "C"): 0.38653123552283214,
    },
    emission_probs={
        ("A", "lo"): 0.19949219710155375,
        ("A", "mid"): 0.30789837305397333,
        ("A", "hi"): 0.492609429844473,
        ("B", "lo"): 0.534907622618408,
        ("B", "mid"): 0.234417585356662,
        ("B", "hi"): 0.23067479202493,
        ("C", "lo"): 0.09093879934300991,
        ("C", "mid"): 0.008996844382398088,
        ("C", "hi"): 0.9000643562745919,
    },
    initialize=True,
)

observed = ["hi", "mid", "lo", "lo", "lo", "lo", "lo"]


def forbid_mvr(state, time_range=None):
    """MVR rejecting any path that visits ``state``."""
    mediation_states = ["ok", "violated"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("violated" if h == state else "ok") for h in HIDDEN_STATES},
        upd={
            (m, h): ("violated" if m == "violated" or h == state else "ok")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"ok": True, "violated": False},
        time_range=time_range,
    )


def reach_mvr(state, time_range=None):
    """MVR accepting once ``state`` has been visited; acceptance is absorbing."""
    mediation_states = ["not_yet", "seen"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("seen" if h == state else "not_yet") for h in HIDDEN_STATES},
        upd={
            (m, h): ("seen" if m == "seen" or h == state else "not_yet")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"not_yet": False, "seen": True},
        time_range=time_range,
    )


def no_repeat_mvr(state):
    """MVR rejecting any path in which ``state`` immediately follows itself."""
    mediation_states = ["ok", "armed", "violated"]

    upd = {}
    for m in mediation_states:
        for h in HIDDEN_STATES:
            if m == "violated" or (m == "armed" and h == state):
                upd[(m, h)] = "violated"
            else:
                upd[(m, h)] = "armed" if h == state else "ok"

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("armed" if h == state else "ok") for h in HIDDEN_STATES},
        upd=upd,
        evl={"ok": True, "armed": True, "violated": False},
    )


# The candidate pool. Deliberately mixes global, local and windowed constraints, and
# includes pairs that contradict each other (forbid A vs reach A by 10).
NAMES = [
    "forbid A",
    "forbid B",
    "no CC",
    "reach B by 5",
    "forbid C early",
    "reach A by 10",
]
CANDIDATES = [
    forbid_mvr("A"),
    forbid_mvr("B"),
    no_repeat_mvr("C"),
    reach_mvr("B", time_range=[0, 5]),
    forbid_mvr("C", time_range=[0, 20]),
    reach_mvr("A", time_range=[0, 10]),
]


def model_for(indices):
    """An MVR_CHMM enforcing the candidates at ``indices``."""
    return MVR_CHMM(
        hidden_markov_model=hmm,
        constraints=[CANDIDATES[i] for i in sorted(indices)],
    )


def show(indices):
    return "{" + ", ".join(NAMES[i] for i in sorted(indices)) + "}"


print(f"{len(CANDIDATES)} candidates -> {2 ** len(CANDIDATES)} possible models")

## 1. The score, and the identity behind it

A model's score is its evidence plus a prior over model size:

$$\log S(A) \;=\; \underbrace{\log P(D \mid A)}_{\texttt{log\_evidence\_mvr\_chmm}}
   \;+\; \underbrace{|A|\log\pi + (n-|A|)\log(1-\pi)}_{\text{Bernoulli inclusion prior}}$$

where $\log P(D \mid A) = \log P(D, A \text{ sat}) - \log P(A \text{ sat})$, both terms
being one forward pass — the second over an *empty* observation set.

`bayes_factor_mvr_chmm` computes the difference between two neighbouring models
without ever forming either evidence separately. The two must agree, and they do:
checked below against each other and against brute-force enumeration of all $3^7$
paths.

In [ ]:
base = [0]          # forbid A
candidate = 2       # no CC

direct = log_evidence_mvr_chmm(
    model_for(base + [candidate]), observed
) - log_evidence_mvr_chmm(model_for(base), observed)

factor = bayes_factor_mvr_chmm(model_for(base + [candidate]), observed, target=-1)

print(f"log P(D|{show(base + [candidate])}) - log P(D|{show(base)})")
print(f"   evidence difference = {direct:.12f}")
print(f"   Bayes factor        = {factor:.12f}")
print(f"   agree               = {abs(direct - factor) < 1e-9}")


def brute_log_evidence(indices):
    """log P(D, sat A) - log P(sat A) by enumerating every hidden path."""
    repn = hmm.repn
    joint = prior = 0.0

    for path in itertools.product(HIDDEN_STATES, repeat=len(observed)):
        feasible = True
        for mvr in (CANDIDATES[i] for i in sorted(indices)):
            a, b = mvr._time_range or (0, len(observed) - 1)
            m = mvr.ini[path[a]]
            for t in range(a + 1, b + 1):
                m = mvr.upd[(m, path[t])]
            feasible = feasible and mvr.evl[m]

        if not feasible:
            continue

        idx = [hmm.hidden_to_internal[h] for h in path]
        weight = repn.start_vec[idx[0]]
        for t in range(1, len(idx)):
            weight *= repn.transition_mat[idx[t - 1]][idx[t]]

        prior += weight
        for t, o in enumerate(observed):
            weight *= repn.emission_mat[idx[t]][hmm.observed_to_internal[o]]
        joint += weight

    return math.log(joint) - math.log(prior)


print("\nlog_evidence_mvr_chmm vs enumeration over all 3^7 paths:")
for indices in ([], [0], [0, 2], [2, 3]):
    got = log_evidence_mvr_chmm(model_for(indices), observed)
    want = brute_log_evidence(indices)
    print(f"   {show(indices):34s} {got:10.6f}  vs {want:10.6f}   {abs(got - want) < 1e-9}")

## 2. Data from a known $A^*$

To test recovery we need $D \sim P(\cdot \mid A^*)$. That is two existing pieces: FFBS
with an **empty** observation set draws a hidden path from the *constrained prior*
$P(X \mid A^* \text{ sat})$, and the emission matrix then turns it into observations.

Take $A^* = \{\texttt{forbid A}\}$ over a horizon of 200. A single short sequence
carries almost no signal about which constraints are active, so this is the one place
the notebook departs from the shared 7-step `observed` used above.

In [ ]:
TRUTH = frozenset({0})
T = 200


def generate_data(seed, truth=TRUTH, horizon=None):
    """One hidden path and observed sequence drawn from P(. | truth)."""
    hidden = ffbs_torch_mvr_chmm(
        model_for(truth),
        {},                              # no observations: draw from the constrained prior
        num_samples=1,
        time_horizon=horizon or T,
        generator=torch.Generator().manual_seed(seed),
    )[0]

    # load_model sorts the observed labels, so index the emission columns through
    # observed_to_external rather than through OBSERVED_STATES.
    emission = hmm.repn.emission_mat
    rng = np.random.default_rng(seed)

    return hidden, [
        hmm.observed_to_external[
            rng.choice(len(OBSERVED_STATES), p=np.array(emission[hmm.hidden_to_internal[h]]))
        ]
        for h in hidden
    ]


hidden_path, data = generate_data(5)

print(f"A* = {show(TRUTH)}")
print(f"hidden  {''.join(hidden_path[:60])} ...")
print(f"visits A: {'A' in hidden_path}   (forbidden, so never)")
print(f"observed {' '.join(data[:14])} ...")

## 3. Three things the Bayes factor gets right for free

The score is a genuine log-likelihood ratio, so its degenerate cases are meaningful
rather than special-cased:

| situation | Bayes factor | why |
| --- | --- | --- |
| the candidate is **implied** by the base | exactly $0$ | the model is unchanged, so the ratio is $1$ |
| the candidate **contradicts** the base | $-\infty$ | $P(C \mid B) = 0$, so the model has no feasible path |
| the candidate genuinely restricts | finite, usually negative | it removes paths, some of which fit $D$ |

The first is worth dwelling on. `reach B by 5` is *logically implied* by
`forbid A` $\wedge$ `no CC`: with `A` excluded and no two consecutive `C`, a `B` must
occur within any two steps. The search therefore gets no credit for adding it — and
under a sparsity prior, actively pays for it.

In [ ]:
def bf(base, candidate):
    return bayes_factor_mvr_chmm(model_for(list(base) + [candidate]), data, target=-1)


print(f"implied      bf(reach B by 5 | forbid A, no CC) = {bf([0, 2], 3)}")
print(f"contradicts  bf(reach A by 10 | forbid A)       = {bf([0], 5)}")
print(f"contradicts  bf(reach B by 5  | forbid B)       = {bf([1], 3)}")
print(f"restricts    bf(no CC | forbid A)               = {bf([0], 2):.4f}")
print(f"restricts    bf(forbid C early | forbid A)      = {bf([0], 4):.4f}")

## 4. The search

Each iteration scores every one-constraint neighbour of the current model — $n$
evaluations, two forward passes each — then follows the paper's two-stage move: sample
one representative from the additions and one from the deletions, each in proportion to
score *within its own set*, then sample between those two. Splitting the draw is what
lets the model shrink; sampling the neighbourhood in one pass would almost always grow
it, because there are more additions than deletions.

Every model scored on the way is recorded, so the elite set ranks far more models than
the chain visits. `inclusion_prob` is the sparsity knob: $\pi = 0.5$ is the pure
likelihood.

In [ ]:
result = sss_torch_mvr_chmm(
    hmm,
    CANDIDATES,
    data,
    num_iterations=25,
    inclusion_prob=0.3,
    rng=np.random.default_rng(5),
)

print(f"truth     {show(TRUTH)}")
print(f"recovered {show(result.best)}   score {result.best_score:.3f}")
print(f"exact hit {result.best == TRUTH}\n")

print("elite set (top 6 of every model scored):")
for model, score in result.elite[:6]:
    marker = "  <- truth" if model == TRUTH else ""
    print(f"   {score:10.3f}   {show(model)}{marker}")

visited = {tuple(sorted(step.model)) for step in result.trace}
print(f"\nchain visited {len(visited)} distinct models; elite ranks {len(result.elite)}")

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4))

scores = [step.score for step in result.trace]
left.plot(scores, marker="o", ms=4, color="#3b6ea5")
left.axhline(result.best_score, ls="--", lw=1, color="#c1442e", label=f"best = {show(result.best)}")
left.set_xlabel("iteration")
left.set_ylabel("log S(A)")
left.set_title("Score along the chain")
left.legend(loc="lower right", fontsize=8)

top = result.elite[:8][::-1]
labels = [show(m) for m, _ in top]
values = [s for _, s in top]
colors = ["#c1442e" if m == TRUTH else "#9bb7d4" for m, _ in top]
right.barh(range(len(top)), values, color=colors)
right.set_yticks(range(len(top)), labels, fontsize=8)
right.set_xlim(min(values) - 0.5, max(values) + 0.3)
right.set_xlabel("log S(A)")
right.set_title("Elite set (truth in red)")

fig.tight_layout()
plt.show()

## 5. The one thing the likelihood cannot see

Look again at the trace: the chain does not settle, it **oscillates** between
$\{\texttt{forbid A}\}$ and $\{\texttt{forbid A}, \texttt{reach B by 5}\}$. That is not
a search failure — it is the search reporting, correctly, that those two models are
almost indistinguishable, and it is worth understanding why.

The likelihood is self-regularizing: for $D \sim P(\cdot \mid A^*)$,
$\mathbb{E}_D\, l(A^*) \ge \mathbb{E}_D\, l(A)$ by non-negativity of the KL divergence,
so in principle no complexity penalty is needed at all. Measured over 40 independent
sequences at $T = 200$ — the mean gap $l(A^*) - l(A)$, and how often the truth wins:

| rival | mean gap | truth wins |
| --- | --- | --- |
| a *wrong* model (`forbid B`) | $9.84$ | $100\%$ |
| the *empty* model | $8.17$ | $98\%$ |
| a *restrictive* superset ($A^*$ + `no CC`) | $31.23$ | $100\%$ |
| a *restrictive* superset ($A^*$ + `forbid C early`) | $11.26$ | $100\%$ |
| a **near-vacuous** superset ($A^*$ + `reach B by 5`) | $-0.02$ | $28\%$ |

Four of the five are settled decisively, including supersets. The exception is the one
whose extra constraint **barely changes the distribution**: under `forbid A` the path is
`B`/`C` only, so `reach B by 5` is satisfied by almost every remaining path and its
Bayes factor is $+0.065$ — indistinguishable from zero.

This is not a defect of the search but a property of the objective: a constraint that
excludes nothing costs nothing, so the likelihood has no grounds to reject it. Charging
a fixed price per constraint is exactly what the inclusion prior does, and the two
Bayes factors above say precisely where $\pi$ has to sit: the prior odds
$\log \frac{\pi}{1-\pi}$ must fall between $-2.96$ (the true constraint's gain, which we
must not overturn) and $-0.065$ (the vacuous one's).

In [ ]:
# Recovery is a property of the data, not of the search randomness, so this
# sweeps independent sequences rather than independent search seeds.
PIS = [0.5, 0.4, 0.2, 0.1, 0.05, 0.03, 0.02]
DATASETS = [generate_data(seed)[1] for seed in range(6)]

recovered, sizes = [], []

for pi in PIS:
    picked = [
        sss_torch_mvr_chmm(
            hmm,
            CANDIDATES,
            sequence,
            num_iterations=15,
            inclusion_prob=pi,
            rng=np.random.default_rng(0),
        ).best
        for sequence in DATASETS
    ]
    hits = sum(model == TRUTH for model in picked)
    recovered.append(hits / len(DATASETS))
    sizes.append(np.mean([len(model) for model in picked]))
    print(f"pi={pi:5.2f}   exact recovery {hits}/{len(DATASETS)}   mean |A| = {sizes[-1]:.2f}")

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

left.plot(PIS, recovered, marker="o", color="#3b6ea5")
left.set_ylim(-0.05, 1.05)
left.set_ylabel(f"exact recovery of {show(TRUTH)}")
left.set_title("Recovery rate over 6 sequences")

right.plot(PIS, sizes, marker="s", color="#c1442e")
right.axhline(len(TRUTH), ls=":", lw=1, color="grey")
right.set_ylim(-0.1, 2.2)
right.set_ylabel("mean |A| selected")
right.set_title(f"Selected model size (|A*| = {len(TRUTH)}, dotted)")

for ax in (left, right):
    ax.set_xscale("log")
    ax.set_xticks(PIS, [str(pi) for pi in PIS], fontsize=8)
    ax.set_xlabel("inclusion_prob (log scale)")
    ax.axvspan(0.05, 0.4, color="#3b6ea5", alpha=0.08)

fig.suptitle("Too weak keeps a vacuous constraint; too strong drops the real one")
fig.tight_layout()
plt.show()

The plateau sits exactly where the two Bayes factors said it would. At
$\pi = 0.5$ the prior odds are $0$ and the vacuous `reach B by 5` is kept for free; by
$\pi \approx 0.03$ the odds have passed $-2.96$ and the *real* constraint starts being
penalized away too. Anywhere between, recovery is exact.

Note this is a **wide** plateau — two orders of magnitude in $\pi$ — because the true
constraint's Bayes factor and the vacuous one's differ by a factor of 45. The tuning is
only delicate when a real constraint is itself nearly vacuous, which is the same thing
as saying it barely matters.

**Where to go next.** The binding limit is *one sequence*, not the method. Both terms
above are per-dataset quantities: $N$ independent sequences scale the gap by $N$ while
the noise grows only as $\sqrt{N}$, so with a batch the likelihood would separate even
the vacuous superset and $\pi$ could stay at $0.5$. Summing the evidence over sequences
is the natural next extension of `sss_torch_mvr_chmm`.

Two directions the module deliberately leaves open: the paper's **replacement** moves
$\gamma^\circ$, dropped here because add/delete alone keeps every score exact and the
iteration at $O(nT)$; and multi-constraint **lookahead**, for a block that is worth
adding when no single piece of it is.